# 02 · Benchmark: questions and reference answers

The first version of the natural-language benchmark (point 5 of Myles's week-2 instructions): 15 simple spatial
questions. For each one the table records the wording, the data, the inputs, the operation, the parameters, the
assumptions and how the answer is calculated. The reference answers are computed **here, with the tested
operations** of `src/space_mel/operations.py`, never by an LLM.

**Status of each question**

| Status | Meaning |
|---|---|
| answered | reference answer computed below |
| awaiting annotation | needs real nevus/melanoma labels, which we don't have |
| operation not implemented | needs "signature enrichment", not built yet |

**Data**
- **Toy grid**: a tiny drawn piece of the Visium grid (spots 100 µm apart), answers can be checked by eye.
- **A24**: the real patient, filtered spots. Two sets defined by **one gene each** (expression only, no biology claim):
  **PRAME-high** = PRAME > 1.5 and **CXCL9-high** = CXCL9 > 2 (log-normalised expression).

**One rule for all questions:** "touching" spots = within **150 µm** (between the first and second ring of the
grid; see notebook 01).

In [ ]:
import sys
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
STAGE = ROOT / "02_spatial_operations"
sys.path.insert(0, str(ROOT / "src"))
from space_mel.operations import distance_to_region, neighbourhood_fraction, spots_table, spots_within_radius
from space_mel.spatial import grid_to_um

FIGS = STAGE / "results"
OUT = STAGE / "benchmark"
OUT.mkdir(exist_ok=True)
BLUE, ORANGE, GREY, INK = "#2a78d6", "#eb6834", "#d9d9d6", "#52514e"
TOUCHING_UM = 150
answers = {}          # question id -> reference answer, filled in below

## The 15 questions

In [ ]:
Q = [
    # --- neighbours (spots within a radius) ---
    dict(id="B01", type="neighbours", question="Which spots are within 100 µm of spot r0c0?", data="toy grid",
         inputs="spot r0c0", operation="spots_within_radius", parameters="radius 100 µm", assumptions="none",
         how="list the spots at <= 100 µm from r0c0 (not r0c0 itself)", status="answered"),
    dict(id="B02", type="neighbours", question="How many neighbours does the spot 'alone' have within 100 µm?",
         data="toy grid", inputs="spot alone", operation="spots_within_radius", parameters="radius 100 µm",
         assumptions="none", how="count the spots at <= 100 µm (0 is a valid answer)", status="answered"),
    dict(id="B03", type="neighbours", question="How many spots are within 300 µm of spot AACTGCCGCTACATCT-1?",
         data="A24", inputs="spot AACTGCCGCTACATCT-1", operation="spots_within_radius", parameters="radius 300 µm",
         assumptions="none", how="count the spots at <= 300 µm (not the spot itself)", status="answered"),
    dict(id="B04", type="neighbours", question="Which spots touch spot AACTGCCGCTACATCT-1?", data="A24",
         inputs="spot AACTGCCGCTACATCT-1", operation="spots_within_radius", parameters="radius 150 µm ('touching')",
         assumptions="touching = within 150 µm", how="list the spots at <= 150 µm", status="answered"),
    # --- distance to a region ---
    dict(id="B05", type="distance", question="How far is spot r2c4 from region R?", data="toy grid",
         inputs="spot r2c4; region R = r-2c-4, r-2c-2, r-1c-3", operation="distance_to_region", parameters="µm",
         assumptions="none", how="distance from r2c4 to the nearest spot of R", status="answered"),
    dict(id="B06", type="distance",
         question="What is the median distance from the CXCL9-high spots to the nearest PRAME-high spot?",
         data="A24", inputs="from: CXCL9-high spots; region: PRAME-high spots", operation="distance_to_region",
         parameters="µm", assumptions="PRAME-high = PRAME > 1.5; CXCL9-high = CXCL9 > 2",
         how="distance of every CXCL9-high spot to the nearest PRAME-high spot, then the median", status="answered"),
    dict(id="B07", type="distance",
         question="How many CXCL9-high spots are more than 500 µm away from every PRAME-high spot?", data="A24",
         inputs="from: CXCL9-high spots; region: PRAME-high spots", operation="distance_to_region",
         parameters="threshold 500 µm", assumptions="PRAME-high = PRAME > 1.5; CXCL9-high = CXCL9 > 2",
         how="count the CXCL9-high spots whose distance to the region is > 500 µm", status="answered"),
    dict(id="B08", type="distance",
         question="How far is spot s1 (section A22) from region {s2} (section A24)?", data="two-section toy",
         inputs="spot s1; region = s2, 10 µm away but in another section", operation="distance_to_region",
         parameters="µm", assumptions="sections never connect",
         how="no region spot in s1's section, so the distance is not defined (NaN)", status="answered"),
    dict(id="B09", type="distance", question="How far are the CXCL9-high spots from the nevus in A24?",
         data="A24", inputs="from: CXCL9-high spots; region: nevus", operation="distance_to_region",
         parameters="µm", assumptions="needs a nevus label per spot",
         how="as B06, with the nevus as the region", status="awaiting annotation"),
    # --- neighbourhood composition ---
    dict(id="B10", type="neighbourhood",
         question="What fraction of the neighbours of spot r0c0 (within 100 µm) are in category C?",
         data="toy grid", inputs="spot r0c0; category C = r0c2, r1c1, r-1c1, r1c-1",
         operation="neighbourhood_fraction", parameters="radius 100 µm", assumptions="none",
         how="neighbours in C / all neighbours (r0c0 not counted)", status="answered"),
    dict(id="B11", type="neighbourhood",
         question="What percentage of the PRAME-high spots touch at least one CXCL9-high spot?", data="A24",
         inputs="spots: PRAME-high; category: CXCL9-high", operation="neighbourhood_fraction",
         parameters="radius 150 µm ('touching')", assumptions="PRAME-high = PRAME > 1.5; CXCL9-high = CXCL9 > 2",
         how="for each PRAME-high spot count CXCL9-high neighbours; % of PRAME-high spots with at least 1",
         status="answered"),
    dict(id="B12", type="neighbourhood",
         question="How many PRAME-high spots have at least half of their touching spots CXCL9-high?", data="A24",
         inputs="spots: PRAME-high; category: CXCL9-high", operation="neighbourhood_fraction",
         parameters="radius 150 µm; fraction >= 0.5", assumptions="PRAME-high = PRAME > 1.5; CXCL9-high = CXCL9 > 2",
         how="count PRAME-high spots whose CXCL9-high fraction is >= 0.5", status="answered"),
    dict(id="B13", type="neighbourhood",
         question="What share of the spots along the melanoma border are CXCL9-high?", data="A24",
         inputs="spots: melanoma border; category: CXCL9-high", operation="neighbourhood_fraction",
         parameters="radius 150 µm", assumptions="needs a melanoma label per spot to find its border",
         how="border = melanoma spots touching non-melanoma spots; % of them that are CXCL9-high",
         status="awaiting annotation"),
    # --- signature enrichment ---
    dict(id="B14", type="signature enrichment",
         question="Is CXCL9 higher in spots within 200 µm of the PRAME-high spots than in spots further away?",
         data="A24", inputs="region: PRAME-high; gene: CXCL9", operation="distance_to_region + enrichment test",
         parameters="200 µm", assumptions="PRAME-high = PRAME > 1.5",
         how="compare CXCL9 between near and far spots (test not chosen yet)", status="operation not implemented"),
    dict(id="B15", type="signature enrichment",
         question="Is CXCL9 higher next to the nevus than next to the melanoma?", data="A24",
         inputs="regions: nevus, melanoma; gene: CXCL9", operation="distance_to_region + enrichment test",
         parameters="150 µm", assumptions="needs nevus and melanoma labels",
         how="compare CXCL9 in spots touching each region",
         status="awaiting annotation; operation not implemented"),
]
questions = pd.DataFrame(Q).set_index("id")
questions.groupby(["type", "status"], sort=False).size().rename("questions").to_frame()

## The data

### Toy grid
The same drawing as notebook 01: a piece of the Visium grid plus one isolated spot. Region **R** (orange rings, left)
and category **C** (orange rings, right). Spot names are `r<row>c<column>`.

In [ ]:
rows, cols = np.meshgrid(np.arange(-2, 3), np.arange(-4, 5), indexing="ij")
keep = (rows + cols) % 2 == 0                         # Visium uses every other (row, col) position
xy = grid_to_um(rows[keep], cols[keep])
toy = pd.DataFrame({"x_um": xy[:, 0], "y_um": xy[:, 1], "section": "toy"},
                   index=[f"r{r}c{c}" for r, c in zip(rows[keep], cols[keep])])
toy.loc["alone"] = [700.0, 0.0, "toy"]

R = pd.Series(toy.index.isin(["r-2c-4", "r-2c-2", "r-1c-3"]), index=toy.index)
C = pd.Series(toy.index.isin(["r0c2", "r1c1", "r-1c1", "r1c-1"]), index=toy.index)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, ring, name in zip(axes, [R, C], ["region R", "category C"]):
    ax.scatter(toy.x_um, toy.y_um, s=900, c=GREY, linewidths=0)
    ax.scatter(toy.x_um[ring], toy.y_um[ring], s=900, facecolors="none", edgecolors=ORANGE, linewidths=2.5)
    for b in toy.index:
        ax.annotate(b, (toy.x_um[b], toy.y_um[b]), ha="center", va="center", fontsize=7)
    ax.set_xlim(-480, 780); ax.set_ylim(250, -250)       # y downwards, room for the large markers
    ax.set_aspect("equal"); ax.axis("off")
    ax.set_title(f"Toy grid: {name} (orange rings)", loc="left")
fig.savefig(FIGS / "benchmark_toy_grid.png", bbox_inches="tight")

### A24: two sets defined by one gene each

In [ ]:
a24 = ad.read_h5ad(ROOT / "data/processed/GSE298774_A24.h5ad")   # filtered spots
spots = spots_table(a24)

def gene(name):
    return pd.Series(a24[:, name].X.toarray().ravel(), index=a24.obs_names)

prame_high = gene("PRAME") > 1.5
cxcl9_high = gene("CXCL9") > 2
print(f"A24: {len(spots)} spots | PRAME-high: {prame_high.sum()} | CXCL9-high: {cxcl9_high.sum()} "
      f"| both: {(prame_high & cxcl9_high).sum()}")

fig, ax = plt.subplots(figsize=(6.5, 6.5))
colors = np.select([prame_high & cxcl9_high, cxcl9_high, prame_high], [INK, ORANGE, BLUE], GREY)
ax.scatter(spots.x_um, spots.y_um, c=colors, s=6, linewidths=0)
ax.set_aspect("equal"); ax.invert_yaxis(); ax.axis("off")
ax.set_title("A24: PRAME-high (blue), CXCL9-high (orange), both (dark grey)", loc="left")
fig.savefig(FIGS / "benchmark_A24_sets.png", bbox_inches="tight")

## Reference answers

Each question: the operation(s) to follow, in as few lines as possible.

### Neighbours
**B01**: Which spots are within 100 µm of spot r0c0? (toy grid)

In [ ]:
nb = spots_within_radius(toy, "r0c0", 100)
answers["B01"] = ", ".join(sorted(nb))
print(len(nb), "spots:", answers["B01"])

**B02**: How many neighbours does the spot 'alone' have within 100 µm? (toy grid)

In [ ]:
nb = spots_within_radius(toy, "alone", 100)
answers["B02"] = len(nb)
print(answers["B02"])

**B03**: How many spots are within 300 µm of spot AACTGCCGCTACATCT-1? (A24)

In [ ]:
nb = spots_within_radius(spots, "AACTGCCGCTACATCT-1", 300)
answers["B03"] = len(nb)
print(answers["B03"])

**B04**: Which spots touch spot AACTGCCGCTACATCT-1? (A24, touching = within 150 µm)

In [ ]:
nb = spots_within_radius(spots, "AACTGCCGCTACATCT-1", TOUCHING_UM)
answers["B04"] = ", ".join(sorted(nb))
print(len(nb), "spots:", answers["B04"])

### Distance to a region
**B05**: How far is spot r2c4 from region R? (toy grid)

In [ ]:
d = distance_to_region(toy, R)
answers["B05"] = f"{d['r2c4']:.1f} µm"
print(answers["B05"])

**B06**: What is the median distance from the CXCL9-high spots to the nearest PRAME-high spot? (A24)

In [ ]:
d = distance_to_region(spots, prame_high)          # distance of every spot to the nearest PRAME-high spot
d_cxcl9 = d[cxcl9_high]                            # keep only the CXCL9-high spots
answers["B06"] = f"{d_cxcl9.median():.1f} µm"
print(answers["B06"], f"(over {len(d_cxcl9)} CXCL9-high spots)")

**B07**: How many CXCL9-high spots are more than 500 µm away from every PRAME-high spot? (A24)

In [ ]:
answers["B07"] = int((d_cxcl9 > 500).sum())
print(answers["B07"], "of", len(d_cxcl9))

**B08**: How far is spot s1 (section A22) from region {s2} (section A24)? (two-section toy)

In [ ]:
two = pd.DataFrame({"x_um": [0.0, 10.0], "y_um": [0.0, 0.0], "section": ["A22", "A24"]}, index=["s1", "s2"])
d = distance_to_region(two, pd.Series([False, True], index=two.index))
answers["B08"] = "not defined (NaN): different sections" if np.isnan(d["s1"]) else f"{d['s1']} µm"
print(d.to_dict(), "->", answers["B08"])

### Neighbourhood composition
**B10**: What fraction of the neighbours of spot r0c0 (within 100 µm) are in category C? (toy grid)

In [ ]:
f = neighbourhood_fraction(toy, C, 100)
row = f.loc["r0c0"]
answers["B10"] = f"{int(row.n_in_category)}/{int(row.n_neighbours)} = {row.fraction:.2f}"
print(answers["B10"])

**B11**: What percentage of the PRAME-high spots touch at least one CXCL9-high spot? (A24)

In [ ]:
f = neighbourhood_fraction(spots, cxcl9_high, TOUCHING_UM)   # for every spot: CXCL9-high neighbours
f_prame = f[prame_high]                                       # keep only the PRAME-high spots
answers["B11"] = f"{100 * (f_prame.n_in_category >= 1).mean():.1f}%"
print(answers["B11"], f"({(f_prame.n_in_category >= 1).sum()} of {len(f_prame)} PRAME-high spots)")

**B12**: How many PRAME-high spots have at least half of their touching spots CXCL9-high? (A24)

In [ ]:
answers["B12"] = int((f_prame.fraction >= 0.5).sum())
print(answers["B12"], "of", len(f_prame))

## The benchmark table

Questions B09, B13, B14 and B15 have no answer yet (see their status). The table is saved as
`02_spatial_operations/benchmark/benchmark.csv`.

In [ ]:
questions["reference_answer"] = pd.Series(answers).astype(str)
questions["reference_answer"] = questions["reference_answer"].fillna("-")
questions.to_csv(OUT / "benchmark.csv")
with pd.option_context("display.max_colwidth", 60):
    display(questions[["type", "question", "status", "reference_answer"]])